# Predicting shipping arrivals
### A guided Python + SQL project · synthetic data · 2024–2025

**Your role:** You are a junior analyst at a fictional ocean-visibility company. A logistics team wants to know whether its shipment arrival estimates can be improved, where errors remain large, and what information helps.

**Your question:** Can historical voyage data predict arrival delay more accurately than trusting the schedule?

This project is inspired by the type of problem we discussed for SeaVantage. **It is not SeaVantage's data, model, assessment or recruiting material.** All results describe this simulation only.

You will write the extraction, cleaning, SQL queries, analysis and models yourself. Setup, a tiny worked example and ETL tests are supplied. There is no separate solutions file.

**Only two files are needed, in the same folder:**
- `shipping_arrival_workbook.ipynb` — this notebook.
- `shipping_voyages_2024_2025.xml` — the complete dataset.

The SQLite database stays in memory; charts stay in notebook output. No additional data downloads, API keys, cloud services or project folders are required.

## How to work through this

| Sitting | Sections | Result |
|---|---|---|
| 1 | 0–3 | Extract XML, clean records, create features, pass ETL tests |
| 2 | 4–6 | Explore patterns, query with SQL, compare averages and a simple regression |
| 3 | 7–9 | Add weather, run one hypothesis test, evaluate and explain findings |

- Run cells in order. Fill every **YOUR CODE** / **YOUR OBSERVATION** area before moving on.
- A function with `raise NotImplementedError(...)` is deliberately unfinished. Replace that line with your implementation. An error there is a prompt to work, not a broken installation.
- Use the specified variable names so later cells and tests can find your work.
- Tests protect data rules; they do not grade your writing or guarantee a valid model.
- Save your notebook when you stop. Restarting the kernel clears Python variables and the in-memory database; rerun your completed cells to restore them.
- Complete the planned comparison once on the held-out period. If you later tune models using those results, that period becomes validation data and you would need another test period.

## 0 · Setup

Use Python 3.10+ and a Jupyter-capable environment. In Codespaces, open this notebook, choose a Python kernel, and keep the XML beside it. If notebook support is missing, install the VS Code Python/Jupyter extensions and `ipykernel` in that environment.

Only **four analysis packages** are used:

| Package | Why it is here |
|---|---|
| pandas | Tables, cleaning, timestamps, SQL results |
| NumPy | Absolute values and simple numerical calculations |
| Matplotlib | A few explanatory charts |
| statsmodels | Linear regression, coefficients, confidence intervals and hypothesis tests |

XML parsing, SQL, paths and tests use Python's standard library. Package dependencies install automatically; you do not need to import them separately. Install once while online, then this project runs offline. No automatic package upgrades are requested.

In [ ]:
# Only if an import below fails: uncomment and run this line once.
# %pip install "pandas>=2.0" numpy matplotlib statsmodels

# If your environment cannot offer a Python notebook kernel, run this in its terminal:
# python -m pip install ipykernel

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import sqlite3
import unittest

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATA_PATH = Path("shipping_voyages_2024_2025.xml")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Put the XML beside the notebook, or change DATA_PATH. Current folder: {Path.cwd()}"
    )

pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Setup ready. The next steps are your coding exercises.")

Setup ready. The next steps are your coding exercises.


## 1 · Understand and extract the data

**Grain:** one original row is one voyage, observed exactly **48 hours before its scheduled arrival**. There are 4,800 original voyages, 96 repeated copies, 12 route pairs, four fictional carriers, and 24 months of prediction snapshots. There are 4,896 raw XML records in shuffled order.

All arrival times here mean **arrival at the assigned berth**, not entering the port boundary. A few actual outcomes fall in January 2026 because the snapshots run through December 2025. Unknown outcomes were deliberately included.

Port labels are real; distances and voyage scenarios are approximate fictional teaching data. Do not use them for navigation or conclusions about actual ports.

| Column | Type after cleaning | Meaning / availability |
|---|---|---|
| `voyage_id` | text | Voyage key; repeated copies share the same ID |
| `carrier` | text | Fictional company; known at prediction time |
| `origin_port` / `destination_port` | text | Route endpoints; known at prediction time |
| `prediction_time` | UTC timestamp | When all inputs were captured |
| `scheduled_arrival` | UTC timestamp | Schedule frozen as known at prediction time |
| `total_route_distance_km` | number | Planned sea-route distance, not straight-line distance |
| `distance_remaining_km` | number | Remaining route distance at prediction time |
| `current_speed_knots` | number | Snapshot speed; one knot is one nautical mile per hour |
| `ships_waiting` | whole number | Queue at destination at prediction time; zero is valid |
| `weather_condition` | category | Forecast over the remaining route: calm / moderate / severe |
| `actual_arrival` | UTC timestamp or missing | Outcome known afterward; never a predictor |

**Why two distances?** Total distance describes the journey; remaining distance describes current progress. Longer routes have more scheduled travel time, so longer distance does not automatically mean greater *delay*.

In [2]:
# Supplied: inspect the XML structure and its provenance.
root = ET.parse(DATA_PATH).getroot()
meta = root.find("metadata")
print(meta.findtext("notice"))
print("Snapshot coverage:", meta.findtext("snapshot_start_utc"), "to", meta.findtext("snapshot_end_utc"))
print("Port codes:", {p.get("code"): p.text for p in meta.findall("./ports/port")})
print(ET.tostring(root.find("./voyages/voyage"), encoding="unicode"))

FIELDS = [
    "voyage_id", "carrier", "origin_port", "destination_port",
    "prediction_time", "scheduled_arrival", "total_route_distance_km",
    "distance_remaining_km", "current_speed_knots", "ships_waiting",
    "weather_condition", "actual_arrival",
]
NUMERIC_COLUMNS = ["total_route_distance_km", "distance_remaining_km",
                   "current_speed_knots", "ships_waiting"]
TIME_COLUMNS = ["prediction_time", "scheduled_arrival", "actual_arrival"]
VALID_WEATHER = {"calm", "moderate", "severe"}

Entirely synthetic, not SeaVantage data or model outputs. Port names are real labels; routes, distances, companies, weather, congestion and disruptions are fictional teaching approximations. Not navigation or operational data.
Snapshot coverage: 2024-01-01T00:00:00Z to 2025-12-31T23:59:00Z
Port codes: {'CNSHA': 'Shanghai', 'KRPUS': 'Busan', 'HKHKG': 'Hong Kong', 'SGSIN': 'Singapore', 'AEJEA': 'Jebel Ali', 'INNSA': 'Nhava Sheva', 'USLAX': 'Los Angeles', 'NLRTM': 'Rotterdam', 'USNYC': 'New York / New Jersey'}
<voyage>
      <voyage_id>V02273</voyage_id>
      <carrier>Pacificway</carrier>
      <origin_port>HKHKG</origin_port>
      <destination_port>KRPUS</destination_port>
      <prediction_time>2024-12-05T12:48:00Z</prediction_time>
      <scheduled_arrival>2024-12-07T12:48:00Z</scheduled_arrival>
      <total_route_distance_km>2750.7</total_route_distance_km>
      <distance_remaining_km>1138.5</distance_remaining_km>
      <current_speed_knots>15.3</current_speed_knots>
      <ships

### Task 1A · Extract (E in ETL)

Implement `extract_xml(path)` returning a DataFrame with the 12 `FIELDS` columns. **Keep the raw values as text** at this stage; use `""` for empty tags. Do not remove duplicates or clean labels yet.

Recipe:
1. Parse the file with `ET.parse(path).getroot()`.
2. Iterate through `root.findall("./voyages/voyage")`.
3. Build one dictionary per voyage, using `node.findtext(field) or ""`.
4. Return `pd.DataFrame(records, columns=FIELDS)`.

`metadata` is documentation, not voyage rows. Print the shape, first five rows and duplicate-ID count. Your extraction should give **4,896 rows × 12 columns** and **96 repeated IDs**.

In [3]:
def extract_xml(path):
    root = ET.parse(path).getroot()

    voyages = root.findall('./voyages/voyage')

    data = []

    for voyage in voyages:
        row = {
            'voyage_id':'',
            'carrier':'',
            'origin_port':'',
            'destination_port':'',
            'prediction_time':'',
            'scheduled_arrival':'',
            'total_route_distance_km':'',
            'distance_remaining_km':'',
            'current_speed_knots':'',
            'ships_waiting':'',
            'weather_condition':'',
            'actual_arrival':''
        }

        for attribute in voyage:
            row[attribute.tag] = attribute.text.strip() if attribute.text else ''

        data.append(row)

    return pd.DataFrame(data)

raw = extract_xml(DATA_PATH)
print(raw.shape)
print("Repeated IDs:", raw["voyage_id"].duplicated().sum())
raw.head()

(4896, 12)
Repeated IDs: 96


,voyage_id,carrier,origin_port,destination_port,prediction_time,scheduled_arrival,total_route_distance_km,distance_remaining_km,current_speed_knots,ships_waiting,weather_condition,actual_arrival
0,V02273,Pacificway,HKHKG,KRPUS,2024-12-05T12:48:00Z,2024-12-07T12:48:00Z,2750.7,1138.5,15.3,17,calm,2024-12-07T17:18:00Z
1,V02539,PACIFICWAY,KRPUS,SGSIN,2025-01-15T06:38:00Z,2025-01-17T06:38:00Z,5067.1,807.1,15.7,39,calm,2025-01-17T05:10:00Z
2,V01409,Meridian,SGSIN,AEJEA,2024-08-05T06:09:00Z,2024-08-07T06:09:00Z,6358.0,867.7,12.1,5,calm,2024-08-06T21:27:00Z
3,V03847,Pacificway,CNSHA,NLRTM,2025-08-12T03:42:00Z,2025-08-14T03:42:00Z,21124.2,1327.0,12.9,10,calm,2025-08-15T01:37:00Z
4,V00527,Harborlink,HKHKG,KRPUS,2024-03-25T09:25:00Z,2024-03-27T09:25:00Z,2731.9,923.6,15.3,12,calm,2024-03-27T10:34:00Z


## 2 · Clean, create features, and load

### Task 2A · Transform (T in ETL)

Implement `clean_voyages(raw)` using the rules below. Return a **new DataFrame**, retaining the original 12 columns and resetting its index. Never mutate `raw`.

| Issue | Rule for this exercise |
|---|---|
| Text formatting | Strip outer spaces; carrier to title case, port codes and ID to uppercase, weather to lowercase |
| Repeated voyage ID | Keep the first copy after normalization (copies in this XML agree) |
| Numeric parsing | `pd.to_numeric(..., errors="coerce")`; reject rows with missing or non-finite required numbers |
| Time parsing | `pd.to_datetime(..., utc=True, errors="coerce", format="ISO8601")` |
| Missing identifiers | Reject empty/missing ID, carrier, origin or destination |
| Distance/speed bounds | Total distance > 0; 0 ≤ remaining distance ≤ total; speed > 0 |
| Queue count | Integer ≥ 0; zero is valid, 3.5 is not |
| Weather | Only calm, moderate or severe after normalization |
| Snapshot timing | Required prediction/schedule timestamps; schedule − prediction must equal 48 hours |
| Known outcome | Actual arrival must be later than prediction time; early relative to the *schedule* is valid |
| Unknown outcome | Keep the row with `actual_arrival = NaT`; do not invent an arrival time |

**Scope choice:** We reject missing predictor rows rather than impute them to keep this first project simple. Count what you exclude and acknowledge possible selection bias. Missing outcomes remain useful for tracking but cannot train or score a supervised model.

Hints: start with `df = raw.copy(deep=True)`; use `np.isfinite(df[NUMERIC_COLUMNS]).all(axis=1)`; combine rules with `&` and parentheses. For the known-outcome rule use `actual.isna() | (actual > prediction)`.

In a real pipeline, conflicting duplicate IDs would need investigation and rejected rows would usually be logged. Here the raw XML is retained unchanged for inspection.

In [14]:
def clean_voyages(raw):
    # YOUR CODE, suggested order:
    # 1. Copy; normalize text.
    df = raw.copy(deep=True)
    # 2. Deduplicate by voyage_id.
    df = df.drop_duplicates(
        subset=['voyage_id'],
        keep='first'
    )
    # 3. Parse numeric and timestamp columns.
    for column in NUMERIC_COLUMNS:
        df[column] = pd.to_numeric(df[column],errors='coerce')

    for column in TIME_COLUMNS:
        df[column] = pd.to_datetime(df[column],utc=True,errors='coerce',format='ISO8601')
    
    valid = np.isfinite(df[NUMERIC_COLUMNS]).all(axis=1)
    valid &= df['ships_waiting'] % 1 == 0
    valid &= df['ships_waiting'] > 0
    valid &= df['total_route_distance_km'] > 0
    valid &= df['distance_remaining_km'] >= 0
    valid &= df['distance_remaining_km'] < df['total_route_distance_km']
    valid &= df['current_speed_knots'] > 0

    df= df.loc[valid].copy()

    # 4. Build a Boolean validity mask using every rule above.

    return df.reset_index(drop=True)
    # 5. Return valid rows, with a fresh index.

clean = clean_voyages(raw)
print("Raw rows:", len(raw))
print("Unique raw IDs:", raw["voyage_id"].nunique())
print("Retained rows:", len(clean))
print("Retained rows with unknown outcomes:", clean["actual_arrival"].isna().sum())
clean.head()

Raw rows: 4896
Unique raw IDs: 4800
Retained rows: 4628
Retained rows with unknown outcomes: 58


,voyage_id,carrier,origin_port,destination_port,prediction_time,scheduled_arrival,total_route_distance_km,distance_remaining_km,current_speed_knots,ships_waiting,weather_condition,actual_arrival
0,V02273,Pacificway,HKHKG,KRPUS,2024-12-05 12:48:00+00:00,2024-12-07 12:48:00+00:00,2750.7,1138.5,15.3,17.0,calm,2024-12-07 17:18:00+00:00
1,V02539,PACIFICWAY,KRPUS,SGSIN,2025-01-15 06:38:00+00:00,2025-01-17 06:38:00+00:00,5067.1,807.1,15.7,39.0,calm,2025-01-17 05:10:00+00:00
2,V01409,Meridian,SGSIN,AEJEA,2024-08-05 06:09:00+00:00,2024-08-07 06:09:00+00:00,6358.0,867.7,12.1,5.0,calm,2024-08-06 21:27:00+00:00
3,V03847,Pacificway,CNSHA,NLRTM,2025-08-12 03:42:00+00:00,2025-08-14 03:42:00+00:00,21124.2,1327.0,12.9,10.0,calm,2025-08-15 01:37:00+00:00
4,V00527,Harborlink,HKHKG,KRPUS,2024-03-25 09:25:00+00:00,2024-03-27 09:25:00+00:00,2731.9,923.6,15.3,12.0,calm,2024-03-27 10:34:00+00:00


### Task 2B · Create useful columns

Implement `make_features(clean)` without changing `clean`:

| New column | Calculation | Interpretation |
|---|---|---|
| `actual_delay_hours` | (actual − scheduled).total_seconds() / 3600 | +8 = 8 hours late; −3 = 3 hours early |
| `total_route_distance_1000km` | total distance / 1000 | Coefficient will mean hours per extra 1,000 km |
| `distance_remaining_1000km` | remaining distance / 1000 | Same convenient units |
| `prediction_month` | prediction timestamp formatted as `%Y-%m` | Monthly grouping key |

Use `.dt.total_seconds()` on a pandas timedelta Series. **Do not use `.dt.seconds`**: it drops whole days and behaves differently for negative durations. A missing actual arrival must produce a missing delay, not zero. Delay is our **target (`y`)**, not an input (`X`).

In [ ]:
def make_features(clean):
    # YOUR CODE: copy the table and add the four columns above.
    raise NotImplementedError("Task 2B: implement make_features")

features = make_features(clean)
features[["voyage_id", "actual_delay_hours", "total_route_distance_1000km"]].head()

### Task 2C · Load (L in ETL)

Implement `load_voyages(frame, connection)` to write a SQLite table named `voyages` using pandas `to_sql`.
- Use `if_exists="replace"` so rerunning the cell does not append duplicate rows.
- Use `index=False` so the DataFrame index does not become a data column.
- Load **all retained rows**, including rows with unknown outcomes. These should become SQL `NULL`.

This small database is in memory and is rebuilt from the XML when needed. It creates no extra file. SQLite stores these timestamps as text; we already calculated hours in pandas, so our SQL tasks do not need database-specific date arithmetic.

In [ ]:
def load_voyages(frame, connection):
    # YOUR CODE: one to_sql call; no return value is required.
    raise NotImplementedError("Task 2C: implement load_voyages")

conn = sqlite3.connect(":memory:")
load_voyages(features, conn)
pd.read_sql_query("SELECT COUNT(*) AS loaded_rows FROM voyages", conn)

## 3 · Run the ETL tests

**Run this supplied cell after completing Tasks 1–2.** The tests use tiny independent examples as well as the XML. If a test fails, its name points to a rule to revisit. Fix your transformation, then rerun its function cell and the tests; do not weaken a test to hide a problem.

These tests deliberately check that **zero, missing, and negative delay mean different things**. They also check reruns, SQL NULLs, and that invalid rows are removed without deleting valid voyages.

In [ ]:
def sample_record(voyage_id="T001", **changes):
    row = {
        "voyage_id": voyage_id, "carrier": "Oceanblue",
        "origin_port": "CNSHA", "destination_port": "KRPUS",
        "prediction_time": "2024-01-01T12:00:00Z",
        "scheduled_arrival": "2024-01-03T12:00:00Z",
        "total_route_distance_km": "2000", "distance_remaining_km": "1000",
        "current_speed_knots": "15", "ships_waiting": "0",
        "weather_condition": "calm", "actual_arrival": "2024-01-03T09:00:00Z",
    }
    row.update(changes)
    return row


class TestShippingETL(unittest.TestCase):
    def test_xml_rows_columns_and_duplicates(self):
        extracted = extract_xml(DATA_PATH)
        self.assertEqual(list(extracted.columns), FIELDS)
        self.assertEqual(len(extracted), 4896)
        self.assertEqual(extracted["voyage_id"].nunique(), 4800)
        self.assertTrue(extracted["total_route_distance_km"].map(lambda x: isinstance(x, str)).all())

    def test_text_numeric_and_utc_normalization(self):
        row = sample_record(carrier=" OCEANBLUE ", destination_port=" krpus ",
                            weather_condition=" Severe ", ships_waiting="12")
        result = clean_voyages(pd.DataFrame([row]))
        self.assertEqual(len(result), 1)
        self.assertEqual(result.iloc[0]["carrier"], "Oceanblue")
        self.assertEqual(result.iloc[0]["destination_port"], "KRPUS")
        self.assertEqual(result.iloc[0]["weather_condition"], "severe")
        self.assertEqual(result.iloc[0]["ships_waiting"], 12)
        self.assertEqual(str(result["prediction_time"].dt.tz), "UTC")

    def test_deduplicate_by_id_not_shared_attributes(self):
        data = pd.DataFrame([sample_record("T001"), sample_record("T001"), sample_record("T002")])
        result = clean_voyages(data)
        self.assertEqual(set(result["voyage_id"]), {"T001", "T002"})
        self.assertEqual(len(result), 2)

    def test_zero_queue_and_early_arrival_are_valid(self):
        result = make_features(clean_voyages(pd.DataFrame([sample_record()])))
        self.assertEqual(len(result), 1)
        self.assertEqual(result.iloc[0]["ships_waiting"], 0)
        self.assertAlmostEqual(result.iloc[0]["actual_delay_hours"], -3.0)

    def test_unknown_arrival_stays_missing(self):
        for value in ("", "unknown"):
            with self.subTest(value=value):
                result = make_features(clean_voyages(pd.DataFrame([sample_record(actual_arrival=value)])))
                self.assertEqual(len(result), 1)
                self.assertTrue(pd.isna(result.iloc[0]["actual_arrival"]))
                self.assertTrue(pd.isna(result.iloc[0]["actual_delay_hours"]))

    def test_reject_invalid_inputs_but_keep_valid_row(self):
        bad_changes = [
            {"ships_waiting": ""}, {"ships_waiting": "many"}, {"ships_waiting": "-1"},
            {"ships_waiting": "2.5"}, {"current_speed_knots": "0"},
            {"current_speed_knots": "inf"}, {"total_route_distance_km": "-2"},
            {"distance_remaining_km": "-1"}, {"distance_remaining_km": "2500"},
            {"weather_condition": "unreported"}, {"origin_port": " "},
            {"prediction_time": "bad-time"},
            {"scheduled_arrival": "2024-01-03T11:00:00Z"},
            {"actual_arrival": "2024-01-01T12:00:00Z"},
        ]
        for changes in bad_changes:
            with self.subTest(changes=changes):
                result = clean_voyages(pd.DataFrame([sample_record("GOOD"), sample_record("BAD", **changes)]))
                self.assertEqual(result["voyage_id"].tolist(), ["GOOD"])

    def test_total_seconds_preserves_multi_day_delay(self):
        row = sample_record(actual_arrival="2024-01-05T14:00:00Z")
        result = make_features(clean_voyages(pd.DataFrame([row])))
        self.assertAlmostEqual(result.iloc[0]["actual_delay_hours"], 50.0)
        self.assertAlmostEqual(result.iloc[0]["total_route_distance_1000km"], 2.0)
        self.assertAlmostEqual(result.iloc[0]["distance_remaining_1000km"], 1.0)
        self.assertEqual(result.iloc[0]["prediction_month"], "2024-01")

    def test_transformations_do_not_mutate_inputs(self):
        original = pd.DataFrame([sample_record(carrier=" OCEANBLUE ")])
        before = original.copy(deep=True)
        cleaned = clean_voyages(original)
        pd.testing.assert_frame_equal(original, before)
        before_clean = cleaned.copy(deep=True)
        make_features(cleaned)
        pd.testing.assert_frame_equal(cleaned, before_clean)

    def test_cleaning_is_repeatable(self):
        first = clean_voyages(pd.DataFrame([sample_record(), sample_record()]))
        second = clean_voyages(first)
        pd.testing.assert_frame_equal(first, second)

    def test_sql_load_is_repeatable_and_preserves_null(self):
        frame = make_features(clean_voyages(pd.DataFrame([
            sample_record("T001"), sample_record("T002", actual_arrival="")
        ])))
        with sqlite3.connect(":memory:") as test_conn:
            load_voyages(frame, test_conn)
            load_voyages(frame, test_conn)
            self.assertEqual(test_conn.execute("SELECT COUNT(*) FROM voyages").fetchone()[0], 2)
            self.assertEqual(test_conn.execute("SELECT COUNT(*) FROM voyages WHERE actual_delay_hours IS NULL").fetchone()[0], 1)
            self.assertEqual(test_conn.execute("SELECT MIN(actual_delay_hours) FROM voyages").fetchone()[0], -3.0)
            cols = [r[1] for r in test_conn.execute("PRAGMA table_info(voyages)")]
            self.assertNotIn("index", cols)


test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestShippingETL)
)
assert test_result.wasSuccessful(), "Fix the ETL functions above before continuing."

**YOUR OBSERVATION — ETL audit**

- Raw records / duplicate copies / retained unique voyages: ...
- Unique voyages rejected for invalid required data: ...
- Retained voyages with an unknown outcome: ...
- One cleaning choice that could bias our sample: ...

Do not count duplicates as additional real voyages. Unknown outcomes are not equivalent to on-time arrivals.

## 4 · Separate past learning from future evaluation

Our **training period** is January 2024–June 2025. The **test period** is July–December 2025. We learn coefficients and averages from training voyages, then evaluate on the later period.

Important: a training voyage's outcome must also have happened **before July 1, 2025**. A June 30 snapshot whose actual arrival is July 2 would not have a known label at the cutoff. Exclude such rows from training. This small boundary gap prevents future outcomes leaking into the past.

### Task 4A
1. Set `CUTOFF = pd.Timestamp("2025-07-01", tz="UTC")`.
2. `model_ready`: rows with a known `actual_delay_hours`, sorted by `prediction_time`.
3. `train`: prediction time < cutoff **and actual arrival < cutoff**.
4. `test`: prediction time ≥ cutoff.
5. Call `.copy()` on both selections. Report their row counts and date ranges.

For now, explore only `train`. Use the full dataset for quality/coverage checks, not for selecting models based on future outcomes. We assume historical actual times became available immediately; real systems may need an extra reporting-lag buffer.

In [ ]:
# YOUR CODE: create CUTOFF, model_ready, train and test.

In [ ]:
# Supplied leakage checks; run after Task 4A.
assert len(train) > 0 and len(test) > 0
assert train["prediction_time"].max() < CUTOFF
assert train["actual_arrival"].max() < CUTOFF
assert test["prediction_time"].min() >= CUTOFF
assert set(train["voyage_id"]).isdisjoint(test["voyage_id"])
assert train["actual_delay_hours"].notna().all() and test["actual_delay_hours"].notna().all()
print("Training rows:", len(train), "| Test rows:", len(test))
print("Known-outcome rows excluded at the training boundary:", len(model_ready)-len(train)-len(test))

### Task 4B · Explore with three small views

1. Count training voyages by destination and weather. Are some groups much smaller?
2. Plot training monthly **mean and median delay**, with hours on the y-axis.
3. Plot `total_route_distance_1000km` against `actual_delay_hours` using a scatterplot (`alpha=0.15` helps overlapping points).

**Mean** = sum / count. **Median** = middle value after sorting. A few very long delays can pull the mean above the median. A **sample size (`n`)** is the number of voyages behind a result. Always show it alongside group averages.

Useful tools: `.groupby(...).agg(...)`, `.value_counts()`, `.plot()`, `plt.scatter(...)`. Keep charts titled and label units. A pattern is a question to investigate, not proof that one variable causes another.

In [ ]:
# YOUR CODE: group counts, monthly mean/median chart, distance-vs-delay scatterplot.

**YOUR OBSERVATION — training data**

- A pattern I noticed: ...
- Its group size or time coverage: ...
- Another explanation besides causation: ...
- Something I cannot conclude from this chart: ...

### Task 4C · SQL practice on the same data

Load `train` into a second in-memory table, `training_voyages`, with `index=False` and `if_exists="replace"`. Then write two queries using `pd.read_sql_query(sql, conn)`:

1. By destination: voyage count, average delay, and percentage arriving **more than six hours late**. Show groups with at least 30 voyages and order by average delay descending.
2. By weather: voyage count and average delay.

SQL hints: `COUNT(*)`, `AVG(actual_delay_hours)`, `GROUP BY`, `HAVING`, and `100.0 * AVG(CASE WHEN actual_delay_hours > 6 THEN 1.0 ELSE 0.0 END)`.

The six-hour threshold is a chosen teaching tolerance, not a SeaVantage rule. This **late-arrival rate** differs from the prediction metric **within-six-hours accuracy** introduced next.

In [ ]:
# YOUR CODE: load training_voyages, write and execute both SQL queries.

## 5 · Understand prediction error before modeling

Our target is delay in hours. A model output of `+6` means arrival six hours after the frozen schedule; `-2` means two hours before it.

| Quantity | Calculation | Question answered |
|---|---|---|
| Actual delay | actual arrival − scheduled arrival | Was the ship early or late? |
| Signed prediction error | actual delay − predicted delay | Was our prediction too early or too late? |
| Absolute error | absolute value of signed error | How many hours off were we? |
| MAE | mean absolute error | How far off, on average? |
| Median absolute error | middle absolute error | What was the middle-sized miss? |
| Within-six-hours rate | % with absolute error ≤ 6 | How often was the estimate within our tolerance? |

With our sign convention, **positive signed error = arrival later than predicted** (prediction too optimistic). Opposite signs can cancel, so mean signed error must not replace MAE.

A ship can be very late relative to its schedule but have zero prediction error if we predicted that lateness correctly.

In [ ]:
# Supplied tiny example: not the project results.
toy_actual = np.array([8.0, -2.0, 5.0])
toy_predicted = np.array([6.0, 1.0, 5.0])
toy_signed = toy_actual - toy_predicted
toy_absolute = np.abs(toy_signed)
print("Signed errors:", toy_signed)       # [2, -3, 0]
print("Absolute errors:", toy_absolute)   # [2, 3, 0]
print("MAE:", toy_absolute.mean())        # about 1.67 hours

### Task 5A · Write a reusable scoring function

Implement `score_predictions(actual, predicted)`, returning a dictionary with:
`mae_hours`, `median_abs_error_hours`, `within_6h_pct`, `mean_signed_error_hours`, and `n`.

Convert both inputs with `np.asarray(..., dtype=float)`; verify equal lengths and no missing/non-finite values. Use `.mean()`, `np.median(...)`, and `100 * (... <= 6).mean()`.

Why verify? Silently ignoring missing predictions could make a weak model look better by scoring only its easy cases. All models here must predict every row in the same test set.

In [ ]:
def score_predictions(actual, predicted):
    # YOUR CODE: validate arrays, calculate signed and absolute error, return metrics.
    raise NotImplementedError("Task 5A: implement score_predictions")

toy_score = score_predictions(toy_actual, toy_predicted)
assert np.isclose(toy_score["mae_hours"], 5/3)
assert toy_score["within_6h_pct"] == 100
assert toy_score["n"] == 3
toy_score

## 6 · Build simple predictors, then one regression

### Task 6A · Two baselines

Create a dictionary named `predictions`, where every value is an array or Series of predicted delay hours **in test-row order**.

1. `"Schedule (zero delay)"`: an array of zeroes. This predicts that every voyage arrives at its scheduled time.
2. `"Historical port mean"`: calculate the mean delay per destination using **train only**, then map those means to test destinations. For an unseen destination use the overall training mean.

Hints: `np.zeros(len(test))`; `train.groupby("destination_port")["actual_delay_hours"].mean()`; `test["destination_port"].map(...)`; `.fillna(train["actual_delay_hours"].mean())`.

Do not use test outcomes to calculate the historical averages. A useful regression should earn its complexity by improving on simple alternatives.

In [ ]:
# YOUR CODE: create the predictions dictionary with the two baselines.

### Task 6B · Simple linear regression

Predict delay from **ships currently waiting**:

`predicted delay = intercept + coefficient × ships_waiting`

- **Intercept:** prediction at zero waiting ships.
- **Coefficient / slope:** predicted change in delay for one additional waiting ship.
- **Fit / train:** learn these numbers from historical examples.
- **OLS (ordinary least squares):** chooses the line that minimizes the sum of squared training errors. We can still evaluate it with MAE, which is easier to express in hours.

Statsmodels formula pattern: `smf.ols("target ~ input", data=train).fit()`.

Create `simple_model` for `actual_delay_hours ~ ships_waiting`. Print `.params`, then store `.predict(test)` under `predictions["Queue-only regression"]`. You do not need to interpret every item in `.summary()`.

Plot the training scatter and fitted line. For the line, create `queue_grid = pd.DataFrame({"ships_waiting": np.linspace(..., ..., 100)})` over the training range and predict on it. Avoid drawing a jagged line through unsorted observations.

In [ ]:
# YOUR CODE: fit simple_model, add its test predictions, plot the training scatter and line.

**YOUR OBSERVATION — the first model**

- Intercept: ... hours; queue coefficient: ... hours per extra waiting ship.
- My plain-English interpretation: ...
- Why a slope alone does not establish that congestion caused the delays: ...

## 7 · Add distance, speed and weather

### Task 7A · Two planned multiple regressions

**Multiple regression** means several input variables predicting one number.

Build `base_model` using these four numerical inputs:
- `ships_waiting`
- `total_route_distance_1000km`
- `distance_remaining_1000km`
- `current_speed_knots`

Then build `weather_model` using the **same inputs plus forecast weather**. Use the same training rows for both. Store test predictions under `"Multiple / no weather"` and `"Multiple / with weather"`.

Formula terms are joined by `+`. For weather, append:

`C(weather_condition, Treatment(reference="calm"))`

This automatically creates indicator columns; you do not need scikit-learn or a manual encoder.

| Forecast | Moderate indicator | Severe indicator |
|---|---:|---:|
| calm | 0 | 0 |
| moderate | 1 | 0 |
| severe | 0 | 1 |

A severe-weather coefficient of +7 would mean **seven additional predicted hours compared with calm**, holding the model's other inputs constant. This number is only an example. Coding weather as 0, 1, 2 would impose equally spaced effects that we do not want to assume.

Weather here is a forecast known at prediction time. Observed future weather or the voyage's eventual waiting time would leak future information.

In [ ]:
# YOUR CODE: base_model and weather_model, their parameters, and their test predictions.
# Formula pattern:
# base_formula = "actual_delay_hours ~ input1 + input2 + input3 + input4"
# weather_formula = base_formula + ' + C(weather_condition, Treatment(reference="calm"))'

### Task 7B · One hypothesis test about total distance

**Question:** In the training data, is total route distance linearly associated with arrival delay?

Fit a separate simple model, `distance_model`, with `actual_delay_hours ~ total_route_distance_1000km`. This is an explanatory exercise, not an additional model selected from test performance.

- **H₀ (null):** the distance coefficient is zero.
- **H₁ (alternative):** the distance coefficient differs from zero (two-sided).
- Choose **α = 0.05** before reading the p-value.

Use `.fit(cov_type="HC3")` for this exercise. HC3 adjusts uncertainty estimates for unequal error variability; it does **not** fix omitted variables or dependence between voyages.

Extract the distance coefficient from `.params`, its p-value from `.pvalues`, and its 95% confidence interval from `.conf_int()`.

| Term | Meaning here |
|---|---|
| Coefficient | Estimated change in delay hours per additional 1,000 km |
| p-value | Under H₀ and the test assumptions, how unusual a statistic at least this extreme would be |
| 95% confidence interval | An interval from a method that covers the true coefficient in 95% of repeated samples when its assumptions hold |
| Statistical significance | p < 0.05 under our chosen rule; not automatically a large or useful effect |

**Do not say:** “p = 0.03 means a 3% chance H₀ is true.” If p ≥ 0.05, say “we did not find sufficient evidence of a nonzero linear association,” not “distance has no effect.”

This unadjusted relationship may reflect route mix or destinations. Repeated routes and shared seasonal conditions can also make independent-observation uncertainty estimates too optimistic. Treat this as a learning demonstration; a production study would investigate time/route dependence and appropriate clustered or time-aware inference. A sample of synthetic voyages cannot establish a real-world shipping effect.

In [ ]:
# YOUR CODE: fit distance_model with HC3; report slope, p-value, 95% CI and alpha decision.
# Useful lookup key: "total_route_distance_1000km"

**YOUR OBSERVATION — hypothesis test**

- Estimated association per 1,000 km and confidence interval: ...
- p-value and decision at α = 0.05: ...
- Why this is not proof of causation: ...
- Why a statistically detectable association may still have little predictive value: ...

Optional thinking question: Is the distance coefficient in the multiple regression similar? If not, could accounting for other inputs explain the change? Do not add repeated significance tests simply to search for p < 0.05.

## 8 · Evaluate on the held-out six months

### Task 8A · Compare the five planned approaches

Run `score_predictions(test["actual_delay_hours"], values)` for each entry in `predictions`. Create a DataFrame named `comparison` with one row per approach and columns for the five metrics. Sort by MAE and plot a horizontal MAE bar chart labeled in hours.

All methods must have `n == len(test)`. A lower MAE is better; a higher within-six-hours percentage is better. Mean signed error near zero does not guarantee small errors.

Report the **weather contribution** as MAE(no weather) − MAE(with weather). Positive means weather helped on this holdout; zero/negative means it did not. Do not force the answer you expected.

Because we compare a preplanned set, results are descriptive holdout comparisons. Do not claim the best observed score is a guarantee of future performance or tune repeatedly on this test set.

In [ ]:
# YOUR CODE: comparison table, equal-n check, MAE chart, and weather MAE difference.

### Task 8B · Where does the weather model struggle?

Create `evaluated = test.copy()` and add:
- `predicted_delay_hours` from the weather model.
- `signed_error_hours` = actual delay − predicted delay.
- `absolute_error_hours` = absolute value of signed error.
- `predicted_arrival` = scheduled arrival + `pd.to_timedelta(predicted_delay_hours, unit="h")`.

Then:
1. By destination and separately by weather: report count, MAE, median absolute error and mean signed error. Flag groups below 30 voyages as small samples.
2. Plot the weather model's monthly MAE across the test period. Do any months look less reliable?
3. Plot **signed error versus predicted delay**, with a horizontal zero line (`plt.axhline(0, ...)`). A curve can suggest missing nonlinear structure; widening scatter can indicate unequal error variability. Neither diagnoses a cause by itself.
4. Inspect the five largest absolute errors. What would you investigate before explaining them to a customer?

Show counts and context. Forecast mistakes, unexpected disruption, route differences and missing inputs can all leave error even in a useful model. Correlated inputs can make individual coefficients hard to interpret. Do not drop inconvenient outliers merely to improve the score.

In [ ]:
# YOUR CODE: evaluated table, group metrics, monthly MAE, residual chart, worst misses.

## 9 · Tell the story

Fill this in using **your computed values**, not assumptions about how models should perform. There may be more than one defensible interpretation.

### Your analyst brief (about 200–300 words)

**Problem and scope**  
I evaluated ... synthetic voyages from ... . Each prediction used information captured ... .

**Data quality**  
I removed ... duplicate copies and ... invalid unique voyages. I retained ... unknown outcomes in the clean table but excluded them from supervised evaluation because ... .

**Main result**  
The schedule baseline had MAE ... hours; the historical-average baseline had ... . The weather model had ... . Adding weather changed MAE by ... .

**Where caution is needed**  
The model was less reliable for ... (n = ...). Its signed errors suggest ... . This does not establish ... .

**Distance finding**  
The estimated association was ... hours per 1,000 km (95% CI ...; p = ...). I interpret this as ... within the simulation, subject to ... .

**Next step**  
I would investigate ... because ... . A useful business action might be ... , but this dataset does not measure financial savings.

**Limitations**  
These are generated data, not operational records. Relationships partly reflect the simulation design. Results do not establish SeaVantage performance, actual port reliability, causation or real-world savings. Missing-data exclusions, correlated route characteristics, forecast uncertainty and time dependence also limit interpretation.

### Interview practice
- Explain the difference between delay and prediction error using one voyage.
- Why did you keep zero queue counts and negative delays?
- What did the ETL tests catch that a row-count check would miss?
- Why did you split chronologically and exclude training labels that arrived after the cutoff?
- Why is forecast weather acceptable, but future observed weather is not?
- What does your regression coefficient mean? What does its p-value **not** mean?
- Where did the model fail, and how would you investigate with real data?

Your strongest interview story is a clear, honest explanation of your decisions—not the number of packages or charts.

## Small reference sheet

| Term | Plain-English reminder |
|---|---|
| ETL | Extract raw records, transform them into reliable columns, load into a usable table |
| Feature / X | Information used as an input when predicting |
| Target / y | Outcome you want to predict: delay hours here |
| Baseline | Simple prediction that a more complex model should be compared with |
| Regression | A method for predicting a numerical outcome |
| Residual / signed error | Actual minus predicted; here positive means later than predicted |
| Absolute error | Size of the miss, ignoring direction |
| MAE | Average size of misses, in the target's units |
| Dummy encoding | Separate 0/1 indicators for categories, with a reference category |
| Data leakage | Information used during training/prediction that would not have been available then |
| Holdout | Later observations kept out of model fitting |

**Optional official documentation** (not needed to work offline):
- [pandas timestamp parsing](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [statsmodels formula examples](https://www.statsmodels.org/stable/example_formulas.html)
- [statsmodels robust covariance options](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html)

**Finish:** Save the notebook. Keep these two files together in your repository. Do not describe synthetic results as real shipping evidence. There is no generated database file or extra project folder to commit.